# 1. Library calling

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime
import warnings
import numpy as np
from IPython.display import clear_output
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")
from selenium.webdriver.support.ui import Select
from janitor import xlsx_table
from tqdm import tqdm

# 2. Defining the Product Information and Location

In [9]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
wait=WebDriverWait(driver, 10)
driver.get('https://www.standardbrand.com/en/products/sensors/sensors/crankshaft-sensors')

In [3]:
frame_0 = driver.find_element(By.ID,'eCatFrame')
driver.switch_to.frame(frame_0)

In [4]:
cols =['Sl.No'] 
#cols2 =['Sl.No',"Attributes"] #,'Review_Mentions'
df= pd.DataFrame(columns=cols)
count=0
#df2= pd.DataFrame(columns=cols2)

In [41]:
es=driver.find_elements(By.CLASS_NAME,"basepart-text")
ps=driver.find_elements(By.CLASS_NAME,"partdesc-text")
for e,p in zip(es,ps):
    df.loc[count,'Sl.No']=count
    df.loc[count,'PartNumber']=e.text
    df.loc[count,'Link']="https://www.standardbrand.com/en/ecatalog?partdetail="+e.text
    df.loc[count,'PartName']=p.text
    count=count+1

In [42]:
print(df.shape)
df

(1142, 4)


,Sl.No,PartNumber,Link,PartName
0,0,FJ47,https://www.standardbrand.com/en/ecatalog?part...,Fuel Injector - MFI - New
1,1,FJ68,https://www.standardbrand.com/en/ecatalog?part...,Fuel Injector - MFI - New
2,2,FJ100,https://www.standardbrand.com/en/ecatalog?part...,Fuel Injector - MFI - New
3,3,FJ241,https://www.standardbrand.com/en/ecatalog?part...,Fuel Injector - MFI - New
4,4,FJ258,https://www.standardbrand.com/en/ecatalog?part...,Fuel Injector - Diesel - New
...,...,...,...,...
1137,1137,FJ285T,https://www.standardbrand.com/en/ecatalog?part...,Fuel Injector - MFI - New
1138,1138,FJ308T,https://www.standardbrand.com/en/ecatalog?part...,Fuel Injector - MFI - New
1139,1139,FJ681T,https://www.standardbrand.com/en/ecatalog?part...,Fuel Injector - MFI - New
1140,1140,FJ1383,https://www.standardbrand.com/en/ecatalog?part...,Fuel Injector - Diesel - Remfd


In [46]:
df['Link'][1]

'https://www.standardbrand.com/en/ecatalog?partdetail=FJ68'

# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [10]:
cols1 =['Sl.No'] #,'Review_Mentions'
cols2 =['Sl.No'] #,'Review_Mentions'
df1= pd.DataFrame(columns=cols1)
df2= pd.DataFrame(columns=cols2)
count1=0
count2=0

In [52]:
driver.current_url

'https://www.standardbrand.com/en/ecatalog?partdetail=FJ633'

In [4]:
df=pd.read_excel(r"C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\96_Competitor Data\Standard Parts\Standard_FuelInjector_Details_Filler.xlsx")

In [6]:
df.tail()

,Sl.No,PartNumber,PartName,Link,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
1226,1226,FJ960NX,YTR,https://www.standardbrand.com/en/ecatalog?part...,NaN,NaN,NaN,NaN,NaN,NaN
1227,1227,FJ1225NX,YTR,https://www.standardbrand.com/en/ecatalog?part...,NaN,NaN,NaN,NaN,NaN,NaN
1228,1228,FJ1243NX,YTR,https://www.standardbrand.com/en/ecatalog?part...,NaN,NaN,NaN,NaN,NaN,NaN
1229,1229,FJ226R,YTR,https://www.standardbrand.com/en/ecatalog?part...,NaN,NaN,NaN,NaN,NaN,NaN
1230,1230,FJ713RP4,YTR,https://www.standardbrand.com/en/ecatalog?part...,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df=df[df["PartName"]=="YTR"].reset_index()

In [21]:
from tqdm import tqdm
for i in tqdm(range(len(df))):
    driver.get(df['Link'][i])
    print(i, df['PartNumber'][i])
    frame_0 = driver.find_element(By.ID,'eCatFrame')
    driver.switch_to.frame(frame_0)
    sleep(3)
    try:
        partname=driver.find_element(By.ID,"partDescText").text
        partnumber=driver.find_element(By.CLASS_NAME,"basepart-text").text
        url=driver.current_url
        try:
            driver.find_element(By.CSS_SELECTOR,'[aria-label="Show All Part Specifications"]').click()
        except:
            pass
        ls=driver.find_elements(By.CLASS_NAME,"psLableL")
        vs=driver.find_elements(By.CLASS_NAME,"psLableR")
        for l,v in zip(ls,vs):
            df1.loc[count1,'Sl.No']=count1
            df1.loc[count1,'PartNumber']=partnumber
            df1.loc[count1,'PartName']=partname
            df1.loc[count1,'Link']=url
            df1.loc[count1,l.text]=v.text
        count1=count1+1
        sleep(1)
        cs=driver.find_elements(By.CLASS_NAME,"bgLable")
        for c in cs:
            if "," in c.text:
                for j in range(c.text.count(",")+1):
                    df2.loc[count2,'Sl.No']=count1
                    df2.loc[count2,'PartNumber']=partnumber
                    df2.loc[count2,'PartName']=partname
                    df2.loc[count2,'Link']=url
                    df2.loc[count2,"Make"]=c.text.split(" ",1)[0]
                    df2.loc[count2,"Model"]=c.text.split("(",1)[0].split(" ",1)[1]
                    df2.loc[count2,"Years"]=c.text.split("(")[j+1].split(')')[0]
                    count2=count2+1
            else:
                df2.loc[count2,'Sl.No']=count1
                df2.loc[count2,'PartNumber']=partnumber
                df2.loc[count2,'PartName']=partname
                df2.loc[count2,'Link']=url
                df2.loc[count2,"Make"]=c.text.split(" ",1)[0]
                df2.loc[count2,"Model"]=c.text.split("(",1)[0].split(" ",1)[1]
                df2.loc[count2,"Years"]=c.text.split("(",1)[1].split(")",1)[0]
                count2=count2+1
    except:
        pass
    clear_output(wait=True)


100%|██████████| 31/31 [03:01<00:00,  5.87s/it]


In [23]:
df1

,Sl.No,PartNumber,PartName,Link,Warranty:,New or Reman:,Color/Finish:,Fuel Injection Type:,Fuel Injector Type:,O.E.M. Replacement:,O-Rings Included:,System Type:,Terminal Gender:,Terminal Quantity:,Terminal Type:,Notes:,Hardware included:,Engine ID Code:,Engine Vin Code:
0,0,FJ928,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,"3 years/36,000 mile",Remanufactured,Metal,Diesel,Direct Injection,Yes,Yes,Direct,Male,4,Pin,NaN,NaN,NaN,NaN
1,1,FJ738,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,"3 years/36,000 mile",Remanufactured,Brown,Diesel,Direct Injection,Yes,Yes,Direct,Male,2,Blade,NaN,NaN,NaN,NaN
2,2,FJ927,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,"3 years/36,000 mile",Remanufactured,Metal,Diesel,Direct Injection,Yes,Yes,Direct,Male,4,Pin,NaN,NaN,NaN,NaN
3,3,FJ960,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,"3 years/36,000 mile",Remanufactured,Black,Diesel,Direct Injection,Yes,Yes,Direct,Male,2,Blade,NaN,NaN,NaN,NaN
4,4,FJ933,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,"3 years/36,000 mile",Remanufactured,Metal,NaN,Direct Injection,NaN,NaN,Direct,Male,2,Stud,w/ 5 Hole Injector Nozzle,NaN,NaN,NaN
5,5,FJ1009,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,"3 years/36,000 mile",Remanufactured,Metal,Diesel,Direct Injection,Yes,Yes,Direct,NaN,2,Stud,NaN,NaN,NaN,NaN
6,6,FJ595,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,"3 years/36,000 mile",Remanufactured,Black,Diesel,Direct Injection,Yes,Yes,Direct,Male,2,Blade,NaN,Yes,NaN,NaN
7,7,FJ926,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,"3 years/36,000 mile",Remanufactured,Black,Diesel,Direct Injection,Yes,Yes,Direct,Male,2,Blade,NaN,NaN,NaN,NaN
8,8,FJ962,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,"3 years/36,000 mile",Remanufactured,Black,Diesel,Direct Injection,Yes,NaN,Direct,Male,2,Blade,NaN,NaN,NaN,NaN
9,9,FJ495,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,"3 years/36,000 mile",Remanufactured,Metal,Diesel,Direct Injection,Yes,NaN,Direct,Male,2,NaN,NaN,NaN,LB7,1


In [24]:
df1.columns

Index(['Sl.No', 'PartNumber', 'PartName', 'Link', 'Warranty:', 'New or Reman:',
       'Color/Finish:', 'Fuel Injection Type:', 'Fuel Injector Type:',
       'O.E.M. Replacement:', 'O-Rings Included:', 'System Type:',
       'Terminal Gender:', 'Terminal Quantity:', 'Terminal Type:', 'Notes:',
       'Hardware included:', 'Engine ID Code:', 'Engine Vin Code:'],
      dtype='object')

In [25]:
df1m=pd.melt(df1, id_vars=['Sl.No', 'PartNumber', 'PartName', 'Link'], value_vars=[ 'Warranty:', 'New or Reman:',
       'Color/Finish:', 'Fuel Injection Type:', 'Fuel Injector Type:',
       'O.E.M. Replacement:', 'O-Rings Included:', 'System Type:',
       'Terminal Gender:', 'Terminal Quantity:', 'Terminal Type:', 'Notes:',
       'Hardware included:', 'Engine ID Code:', 'Engine Vin Code:'])

In [26]:
df1m

,Sl.No,PartNumber,PartName,Link,variable,value
0,0,FJ928,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,Warranty:,"3 years/36,000 mile"
1,1,FJ738,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,Warranty:,"3 years/36,000 mile"
2,2,FJ927,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,Warranty:,"3 years/36,000 mile"
3,3,FJ960,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,Warranty:,"3 years/36,000 mile"
4,4,FJ933,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,Warranty:,"3 years/36,000 mile"
...,...,...,...,...,...,...
430,24,FJ1310NX,Fuel Injector - Diesel - New,https://www.standardbrand.com/en/ecatalog?part...,Engine Vin Code:,NaN
431,25,FJ927NX,Fuel Injector - Diesel - New,https://www.standardbrand.com/en/ecatalog?part...,Engine Vin Code:,NaN
432,26,FJ960NX,Fuel Injector - Diesel - New,https://www.standardbrand.com/en/ecatalog?part...,Engine Vin Code:,NaN
433,27,FJ1225NX,Fuel Injector - Diesel - New,https://www.standardbrand.com/en/ecatalog?part...,Engine Vin Code:,NaN


In [27]:
df2.head()

,Sl.No,PartNumber,PartName,Link,Make,Model,Years
0,1,FJ928,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,Ford,E-350 Club Wagon,05-04
1,1,FJ928,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,Ford,E-350 Super Duty,10-04
2,1,FJ928,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,Ford,E-450 Super Duty,10-04
3,1,FJ928,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,Ford,Excursion,05-04
4,1,FJ928,Fuel Injector - Diesel - Remfd,https://www.standardbrand.com/en/ecatalog?part...,Ford,F-250 Super Duty,07-04


In [28]:
for i in range(len(df2)):
    if "-" in df2['Years'][i]:
        #print(i)
        year=df2['Years'].str.split('-')[i]
        startyear=year[1]
        endyear=year[0]
        df2.loc[i,'Start_Year']=startyear
        df2.loc[i,'End_Year']=endyear
    if "-" not in df2['Years'][i]:
        #print (i)
        df2.loc[i,'Start_Year']=df2['Years'][i]
        df2.loc[i,'End_Year']=df2['Years'][i]


In [29]:

Nprefix="20"
Oprefix="19"
for i in range(len(df2)):

    if int(df2['Start_Year'][i]) <26:
        df2.loc[i,'Start_Year']=int(Nprefix+df2['Start_Year'][i])
    else:
        df2.loc[i,'Start_Year']=int(Oprefix+df2['Start_Year'][i])

for i in range(len(df2)):

    if int(df2['End_Year'][i]) <26:
        df2.loc[i,'End_Year']=int(Nprefix+df2['End_Year'][i])
    else:
        df2.loc[i,'End_Year']=int(Oprefix+df2['End_Year'][i])


In [30]:

for i in range (len(df2)):    
    df2.at[i,"Years"]=list(range(df2["Start_Year"][i],(df2["End_Year"][i]+1)))
    
df2=df2.explode('Years')

In [31]:
colss=['Sl.No','PartNumber','PartName','Link','Make','Model','Years']
df2l=df2[colss]
df2l["Application"]=df2l["Years"].astype(str)+" "+df2l["Make"]+" "+df2l["Model"]

In [32]:
OFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\96_Competitor Data\Standard Parts'
product='FuelInjector'

In [33]:
with pd.ExcelWriter(OFolder+'\\'+f'Standard_{product}_Details_1'+'.xlsx') as writer:  # doctest: +SKIP
    df.to_excel(writer,index=False, sheet_name='List')
    df1.to_excel(writer,index=False, sheet_name='Attributes')
    df2l.to_excel(writer,index=False, sheet_name='Coverage')
    df1m.to_excel(writer,index=False, sheet_name='Attribute_Values')

# 99. Archived Codes

In [3]:
dfdd=pd.read_excel(r"C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\96_Competitor Data\Standard Parts\FI_Picked_MMY.xlsx",sheet_name="Sheet1")

In [4]:
dfdd

,Make,Model,Years
0,Chrysler,200,2011
1,Chrysler,200,2012
2,Chrysler,200,2013
3,Chrysler,200,2014
4,Chrysler,200,2015
...,...,...,...
2564,Ford,F-550 Super Duty,2012
2565,Ford,F-550 Super Duty,2013
2566,Ford,F-550 Super Duty,2014
2567,Ford,F-550 Super Duty,2015


In [18]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
wait=WebDriverWait(driver, 10)


In [42]:
frame_0 = wait.until(EC.presence_of_element_located((By.ID,'eCatFrame')))
driver.switch_to.frame(frame_0)

TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF68CE0AD02+56930]
	(No symbol) [0x00007FF68CD7F602]
	(No symbol) [0x00007FF68CC342E5]
	(No symbol) [0x00007FF68CC798ED]
	(No symbol) [0x00007FF68CC79A2C]
	(No symbol) [0x00007FF68CCBA967]
	(No symbol) [0x00007FF68CC9BCDF]
	(No symbol) [0x00007FF68CCB81E2]
	(No symbol) [0x00007FF68CC9BA43]
	(No symbol) [0x00007FF68CC6D438]
	(No symbol) [0x00007FF68CC6E4D1]
	GetHandleVerifier [0x00007FF68D186F8D+3711213]
	GetHandleVerifier [0x00007FF68D1E04CD+4077101]
	GetHandleVerifier [0x00007FF68D1D865F+4044735]
	GetHandleVerifier [0x00007FF68CEA9736+706710]
	(No symbol) [0x00007FF68CD8B8DF]
	(No symbol) [0x00007FF68CD86AC4]
	(No symbol) [0x00007FF68CD86C1C]
	(No symbol) [0x00007FF68CD768D4]
	BaseThreadInitThunk [0x00007FFB7B767344+20]
	RtlUserThreadStart [0x00007FFB7D0426B1+33]


In [37]:
cols =['Sl.No'] 
#cols2 =['Sl.No',"Attributes"] #,'Review_Mentions'
dfex= pd.DataFrame(columns=cols)
countex=0
#df2= pd.DataFrame(columns=cols2)

In [44]:
for i in tqdm(range(1, len(dfdd))):
    Year=dfdd['Years'][i]
    Make=dfdd['Make'][i]
    try:
        int(dfdd['Model'][i])
        Model=int(dfdd['Model'][i])
    except:
        Model=dfdd['Model'][i]
    print(i,Year,Make,Model)
    driver.find_element(By.CSS_SELECTOR,f'[aria-label="{Year}"]').click()
    sleep(6)

    driver.find_element(By.CSS_SELECTOR,f'[aria-label="{Make}"]').click()
    sleep(6)
    driver.find_element(By.CSS_SELECTOR,f'[aria-label="{Model}"]').click()
    sleep(6)

    els=driver.find_elements(By.CSS_SELECTOR,'[class="vehicleEngineFilter checkbox"]')
    s=len(els)

    for sp in range(s):
        sleep(2)
        ells=driver.find_elements(By.CSS_SELECTOR,'[class="vehicleEngineFilter checkbox"]')
        driver.find_element(By.TAG_NAME,'body').send_keys(Keys.PAGE_DOWN)    
        eng=WebDriverWait(ells[sp], 10).until(EC.presence_of_element_located((By.CLASS_NAME,"checkbox-btn"))).text
        sleep(1)
        ells[sp].click()
        sleep(1)
        driver.find_element(By.TAG_NAME,'body').send_keys(Keys.CONTROL + Keys.HOME)
        pns=driver.find_elements(By.CLASS_NAME,'basepart-text')
        for p in pns:
            pn=p.text
            dfex.loc[countex,'Sl.No']=countex
            dfex.loc[countex,'Year']=dfdd['Years'][i]
            dfex.loc[countex,'Make']=dfdd['Make'][i]
            dfex.loc[countex,'Model']=dfdd['Model'][i]
            dfex.loc[countex,'Engine']=eng
            dfex.loc[countex,'PartNumber']=pn
            countex=countex+1            
        sleep(1)   
        clr=driver.find_elements(By.CLASS_NAME,'clear-icon')
        clr[-1].click()
        sleep(2)

    clr=driver.find_elements(By.CLASS_NAME,'clear-icon')
    driver.find_elements(By.CLASS_NAME,'clear-icon')[2].click()
    sleep(2)
    driver.find_elements(By.CLASS_NAME,'clear-icon')[2].click()
    clear_output(wait=True)


  0%|          | 0/2568 [00:00<?, ?it/s]

1 2012 Chrysler 200


  0%|          | 0/2568 [00:06<?, ?it/s]


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[aria-label="Chrysler"]"}
  (Session info: chrome=122.0.6261.129); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF68CE0AD02+56930]
	(No symbol) [0x00007FF68CD7F602]
	(No symbol) [0x00007FF68CC342E5]
	(No symbol) [0x00007FF68CC798ED]
	(No symbol) [0x00007FF68CC79A2C]
	(No symbol) [0x00007FF68CCBA967]
	(No symbol) [0x00007FF68CC9BCDF]
	(No symbol) [0x00007FF68CCB81E2]
	(No symbol) [0x00007FF68CC9BA43]
	(No symbol) [0x00007FF68CC6D438]
	(No symbol) [0x00007FF68CC6E4D1]
	GetHandleVerifier [0x00007FF68D186F8D+3711213]
	GetHandleVerifier [0x00007FF68D1E04CD+4077101]
	GetHandleVerifier [0x00007FF68D1D865F+4044735]
	GetHandleVerifier [0x00007FF68CEA9736+706710]
	(No symbol) [0x00007FF68CD8B8DF]
	(No symbol) [0x00007FF68CD86AC4]
	(No symbol) [0x00

In [34]:
dfex

,Sl.No,Year,Make,Model,Engine,PartNumber
0,0,2011.0,Chrysler,200,200 - 4 Cyl 2.4L,FJ1058
1,1,2011.0,Chrysler,200,200 - 4 Cyl 2.4L,FJ1147
2,2,2011.0,Chrysler,200,200 - 4 Cyl 2.4L,FJ1147RP6
